In [4]:
import pandas as pd
import json

In [3]:
data_df = pd.read_csv("../../data/test/expanded_triplet_evaluations_3.csv")

In [8]:
filted_data_df = data_df[data_df["is_supported"] == True]

In [9]:
len(filted_data_df)

386

In [10]:
# sample 100 rows from filted_data_df
sampled_data_df = filted_data_df.sample(n=100, random_state=42)

In [11]:
print(len(sampled_data_df))
sampled_data_df.head()

100


,doc_id,document,summary,generated_summary,document_triplets,gen_summary_triplets,summary_triplet_subject,summary_triplet_predicate,summary_triplet_object,summary_triplet_full,is_supported,confidence,reasoning,supporting_evidence
532,23606507,The Â£21.5m Castle Mill development at Port Me...,A campaign group has been granted a hearing to...,An environmental group has launched a legal ch...,"[['Castle Mill development', 'located at', 'Po...","[['environmental group', 'launched', 'legal ch...",legal challenge,against,building of student housing,"['legal challenge', 'against', 'building of st...",True,2,The judicial review (legal challenge) targets ...,"[['judicial review', 'based on', 'Oxford City ..."
475,39950399,"According to three separate analyses, a flood ...",Bots appear to be spamming a US regulator's we...,The US government's website has been flooded w...,"[['three separate analyses', 'detected', 'floo...","[['US government', 'owns', 'website'], ['websi...",US government,owns,website,"['US government', 'owns', 'website']",True,1,The FCC is a US government agency (federal com...,"[['Federal Communications Commission', 'abbrev..."
115,35194738,"The ska group tweeted the news: ""It is with de...","John ""Brad"" Bradbury, drummer with The Special...",The Specials have paid tribute to their former...,"[['ska group', 'tweeted', 'news'], ['Brad', 'i...","[['The Specials', 'paid tribute to', 'Brad Bra...",Brad Bradbury,was,former drummer,"['Brad Bradbury', 'was', 'former drummer']",True,2,John 'Brad' Bradbury was drummer for The Speci...,"[[""John 'Brad' Bradbury"", 'was drummer for', '..."
411,31991181,Media playback is not supported on this device...,Newcastle midfielder Jonas Gutierrez says he f...,Newcastle midfielder Jonas Gutierrez says he f...,"[['Jonas Gutierrez', 'is', 'Argentine'], ['Jon...","[['Jonas Gutierrez', 'plays for', 'Newcastle']...",Newcastle,has player,Jonas Gutierrez,"['Newcastle', 'has player', 'Jonas Gutierrez']",True,2,This is the inverse relationship of the text t...,"[['Jonas Gutierrez', 'plays for', 'Newcastle']]"
225,36792661,The Yomper statue is currently located in fron...,Plans to move a statue depicting a Royal Marin...,"More than 1,800 people have signed a petition ...","[['Yomper statue', 'located at', 'Royal Marine...","[['More than 1,800 people', 'signed', 'petitio...",petition,against,plans,"['petition', 'against', 'plans']",True,2,Direct match: The text triplet explicitly stat...,"[['online petition', 'against', 'moving the st..."


In [13]:
# save the sampled data to csv
sampled_data_df.to_csv("../../data/sensitivity_ck/sampled_expanded_triplet_evaluations_3.csv", index=False)

## Removing evidence path from the kg

In [14]:
import ast

def remove_common_triplets(sampled_data_df):
    """
    Remove common items from document_triplets that appear in supporting_evidence.
    
    Parameters:
    -----------
    sampled_data_df : pd.DataFrame
        DataFrame containing 'document_triplets' and 'supporting_evidence' columns
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with modified 'document_triplets' column
    """
    # Create a copy to avoid modifying the original
    df_copy = sampled_data_df.copy()
    
    # Process each row
    for idx, row in df_copy.iterrows():
        # Parse the columns if they are strings
        doc_triplets = row['document_triplets']
        supporting_evidence = row['supporting_evidence']
        
        # Convert strings to lists if needed
        if isinstance(doc_triplets, str):
            try:
                doc_triplets = ast.literal_eval(doc_triplets)
            except:
                doc_triplets = []
        
        if isinstance(supporting_evidence, str):
            try:
                supporting_evidence = ast.literal_eval(supporting_evidence)
            except:
                supporting_evidence = []
        
        # Ensure they are lists
        if not isinstance(doc_triplets, list):
            doc_triplets = []
        if not isinstance(supporting_evidence, list):
            supporting_evidence = []
        
        # Remove common items from document_triplets
        filtered_triplets = [item for item in doc_triplets if item not in supporting_evidence]
        
        # Update the dataframe
        df_copy.at[idx, 'document_triplets'] = filtered_triplets
    
    return df_copy

In [15]:
# Apply the function to remove common triplets
filtered_sampled_df = remove_common_triplets(sampled_data_df)

In [17]:
display(filtered_sampled_df.head())

,doc_id,document,summary,generated_summary,document_triplets,gen_summary_triplets,summary_triplet_subject,summary_triplet_predicate,summary_triplet_object,summary_triplet_full,is_supported,confidence,reasoning,supporting_evidence
532,23606507,The Â£21.5m Castle Mill development at Port Me...,A campaign group has been granted a hearing to...,An environmental group has launched a legal ch...,"[[Castle Mill development, located at, Port Me...","[['environmental group', 'launched', 'legal ch...",legal challenge,against,building of student housing,"['legal challenge', 'against', 'building of st...",True,2,The judicial review (legal challenge) targets ...,"[['judicial review', 'based on', 'Oxford City ..."
475,39950399,"According to three separate analyses, a flood ...",Bots appear to be spamming a US regulator's we...,The US government's website has been flooded w...,"[[three separate analyses, detected, flood of ...","[['US government', 'owns', 'website'], ['websi...",US government,owns,website,"['US government', 'owns', 'website']",True,1,The FCC is a US government agency (federal com...,"[['Federal Communications Commission', 'abbrev..."
115,35194738,"The ska group tweeted the news: ""It is with de...","John ""Brad"" Bradbury, drummer with The Special...",The Specials have paid tribute to their former...,"[[ska group, tweeted, news], [Brad, is, world'...","[['The Specials', 'paid tribute to', 'Brad Bra...",Brad Bradbury,was,former drummer,"['Brad Bradbury', 'was', 'former drummer']",True,2,John 'Brad' Bradbury was drummer for The Speci...,"[[""John 'Brad' Bradbury"", 'was drummer for', '..."
411,31991181,Media playback is not supported on this device...,Newcastle midfielder Jonas Gutierrez says he f...,Newcastle midfielder Jonas Gutierrez says he f...,"[[Jonas Gutierrez, is, Argentine], [Jonas Guti...","[['Jonas Gutierrez', 'plays for', 'Newcastle']...",Newcastle,has player,Jonas Gutierrez,"['Newcastle', 'has player', 'Jonas Gutierrez']",True,2,This is the inverse relationship of the text t...,"[['Jonas Gutierrez', 'plays for', 'Newcastle']]"
225,36792661,The Yomper statue is currently located in fron...,Plans to move a statue depicting a Royal Marin...,"More than 1,800 people have signed a petition ...","[[Yomper statue, located at, Royal Marines Mus...","[['More than 1,800 people', 'signed', 'petitio...",petition,against,plans,"['petition', 'against', 'plans']",True,2,Direct match: The text triplet explicitly stat...,"[['online petition', 'against', 'moving the st..."


In [18]:
filtered_sampled_df.to_csv("../../data/sensitivity_ck/filtered_sampled_expanded_triplet_evaluations_3.csv", index=False)

## Fixes

In [25]:
result_df = pd.read_csv("../../data/sensitivity_ck/expanded_triplet_evaluations_with_fact_eval_3.csv")
result_df.head()

,doc_id,document,summary,generated_summary,document_triplets,gen_summary_triplets,summary_triplet_subject,summary_triplet_predicate,summary_triplet_object,summary_triplet_full,is_supported,confidence,reasoning,supporting_evidence,sensitivity_fact_eval
0,23606507,The Â£21.5m Castle Mill development at Port Me...,A campaign group has been granted a hearing to...,An environmental group has launched a legal ch...,"[['Castle Mill development', 'located at', 'Po...","[['environmental group', 'launched', 'legal ch...",legal challenge,against,building of student housing,"['legal challenge', 'against', 'building of st...",True,2,The judicial review (legal challenge) targets ...,"[['judicial review', 'based on', 'Oxford City ...","[\n{\n ""summary_triple"": [""environmental grou..."
1,39950399,"According to three separate analyses, a flood ...",Bots appear to be spamming a US regulator's we...,The US government's website has been flooded w...,"[['three separate analyses', 'detected', 'floo...","[['US government', 'owns', 'website'], ['websi...",US government,owns,website,"['US government', 'owns', 'website']",True,1,The FCC is a US government agency (federal com...,"[['Federal Communications Commission', 'abbrev...","[\n {\n ""summary_triple"": [""US government""..."
2,35194738,"The ska group tweeted the news: ""It is with de...","John ""Brad"" Bradbury, drummer with The Special...",The Specials have paid tribute to their former...,"[['ska group', 'tweeted', 'news'], ['Brad', 'i...","[['The Specials', 'paid tribute to', 'Brad Bra...",Brad Bradbury,was,former drummer,"['Brad Bradbury', 'was', 'former drummer']",True,2,John 'Brad' Bradbury was drummer for The Speci...,"[[""John 'Brad' Bradbury"", 'was drummer for', '...","[\n {\n ""summary_triple"": [""The Specials"",..."
3,31991181,Media playback is not supported on this device...,Newcastle midfielder Jonas Gutierrez says he f...,Newcastle midfielder Jonas Gutierrez says he f...,"[['Jonas Gutierrez', 'is', 'Argentine'], ['Jon...","[['Jonas Gutierrez', 'plays for', 'Newcastle']...",Newcastle,has player,Jonas Gutierrez,"['Newcastle', 'has player', 'Jonas Gutierrez']",True,2,This is the inverse relationship of the text t...,"[['Jonas Gutierrez', 'plays for', 'Newcastle']]","[\n {\n ""summary_triple"": [""Jonas Gutierre..."
4,36792661,The Yomper statue is currently located in fron...,Plans to move a statue depicting a Royal Marin...,"More than 1,800 people have signed a petition ...","[['Yomper statue', 'located at', 'Royal Marine...","[['More than 1,800 people', 'signed', 'petitio...",petition,against,plans,"['petition', 'against', 'plans']",True,2,Direct match: The text triplet explicitly stat...,"[['online petition', 'against', 'moving the st...","[\n {\n ""summary_triple"": [""More than 1,80..."


In [31]:
# remove unnecessary entries

def clean_fact_eval_res(fact_eval_res, summary_triplet):
    try:
        res_json = json.loads(fact_eval_res)
    except (json.JSONDecodeError, TypeError):
        # If parsing fails, return the original entry as-is
        return fact_eval_res
    
    print(f"Original JSON: {res_json}")
    summary_triplet_str = str(summary_triplet)
    cleaned_res = []
    
    for item in res_json:
        print(f"Checking item: {item}")
        
        # If item is not a dict (e.g., it's a string), keep it as-is
        if not isinstance(item, dict):
            cleaned_res.append(item)
            continue
        
        try:
            if str(item['summary_triple']) == summary_triplet_str:
                cleaned_res.append(item)
        except KeyError:
            cleaned_res.append(item)  # Keep the item if 'summary_triple' key is missing
            # print(f"Key 'summary_triple' not found in item: {item}")
    
    return cleaned_res
   

In [32]:
result_copy = result_df.copy()

In [33]:
for idx, row in result_df.iterrows():
    fact_eval_res = row['sensitivity_fact_eval']
    summary_triplet = row['summary_triplet_full']
    cleaned_res = clean_fact_eval_res(fact_eval_res, summary_triplet)
    result_copy.at[idx, 'sensitivity_fact_eval'] = cleaned_res

Original JSON: [{'summary_triple': ['environmental group', 'launched', 'legal challenge'], 'is_supported': True, 'supporting_evidence': [['Campaign to Protect Rural England', 'seeking', 'judicial review'], ['Campaign to Protect Rural England', 'will present case to', 'High Court judge'], ['Campaign to Protect Rural England', 'court date', '23 October']], 'confidence': 2, 'reasoning': 'Clear direct support: CPRE (environmental group) is seeking judicial review (legal challenge) and will present their case to High Court judge with a scheduled court date.'}, {'summary_triple': ['legal challenge', 'against', 'building of student housing'], 'is_supported': True, 'supporting_evidence': [['Campaign to Protect Rural England', 'seeking', 'judicial review'], ['Castle Mill development', 'cost', '£21.5m'], ['Castle Mill development', 'criticised as', 'ugly']], 'confidence': 2, 'reasoning': "The judicial review (legal challenge) by CPRE is clearly targeting the Castle Mill development (student hous

In [34]:
# save it to csv
result_copy.to_csv("../../data/sensitivity_ck/expanded_triplet_evaluations_with_fact_eval_3_fixed.csv", index=False)

In [38]:
still_true_count = 0
changed_to_false_count = 0

false_id_list = []

for idx, row in result_copy.iterrows():
    doc_id = row['doc_id']
    fact_eval_res = row['sensitivity_fact_eval']
    if isinstance(fact_eval_res, str):
        try:
            fact_eval_res = json.loads(fact_eval_res)
        except json.JSONDecodeError:
            continue  # Skip rows where JSON parsing fails
    
    if isinstance(fact_eval_res, list):
        for item in fact_eval_res:
            if isinstance(item, dict) and 'is_supported' in item:
                if item['is_supported'] == True:
                    still_true_count += 1
                elif item['is_supported'] == False:
                    false_id_list.append(doc_id)
                    changed_to_false_count += 1

In [40]:
print(false_id_list)

[39950399, 37427024, 36269210, 39339718, 37248643, 29771956, 32300397, 38435201, 38537698, 38264402, 38952672, 34911128, 32300397, 38597885, 34631268, 40787080, 34001040, 37732028, 39070183, 39339718, 38781058, 37042796, 37508085, 38128851, 37427024, 35844158, 37624408, 34227252, 36892983, 37066439, 39806255, 38781058, 40787080, 38264402, 36175342, 37624408, 38435201, 36043765, 29771956, 39055714, 37427024, 39806255, 39055714, 38872899, 34227252, 36175342]


In [39]:
print(f"Count of items still true: {still_true_count}")
print(f"Count of items changed to false: {changed_to_false_count}")

Count of items still true: 53
Count of items changed to false: 46


In [37]:
# save a text file with the counts
with open("../../data/sensitivity_ck/fact_eval_counts.txt", "w") as f:
    f.write("Fact Evaluation Counts - Claude Sonnet 4\n")
    f.write(f"File name: expanded_triplet_evaluations_with_fact_eval_3_fixed.csv\n")
    f.write(f"Count of items still true: {still_true_count}\n")
    f.write(f"Count of items changed to false: {changed_to_false_count}\n")